# B2 · Removing noise from images with PCA

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Add Gaussian noise to the digits, reconstruct with `k` components and measure the error with respect to the **clean** image; choose `k` with a criterion that uses only the noisy data (the Marchenko-Pastur edge: the components with an eigenvalue above the expected noise) and
compare with the absolute best `k`; finally verify that a classifier trained on the components is more robust than one on the pixels.

## Dataset

`load_digits`, stratified 60/40 split, with Gaussian noise of standard deviation 4 added to training and test (`X_tr_n`, `X_te_n`; the clean versions `X_tr`, `X_te` serve only to **measure**). The pixels go from 0 to 16.

## Educational purpose

See PCA as a **filter**: the signal (the shapes of the digits) sits in the first components, the noise is spread over all of them, so reconstructing with few components reduces it. And learn a practical criterion to choose `k` without having the clean image.

## Task

1. For `k` in `[2, 5, 10, 15, 20, 25, 30, 40, 64]` train `PCA(k)` on the **noisy training set**, reconstruct the noisy test set and compute the mean squared error with respect to the **clean** test set: save `mse_per_k` (`pd.Series`), `oracle_k` (the best) and `noisy_mse` (error of the noisy image without reconstruction).
2. **Criterion without oracle:** with `sigma = 4`, `n, p = X_tr_n.shape` and `threshold = sigma**2 * (1 + np.sqrt(p / n))**2` (the maximum edge of the eigenvalues of the noise alone), save `mp_k` = number of components with `explained_variance_ > threshold` of the complete PCA on the noisy training set. Save `mp_mse` (error with `mp_k` components) and `reduction` = `1 - mse_per_k[oracle_k] / noisy_mse`.
3. **Classification:** train `LogisticRegression(max_iter=3000)` on the noisy pixels of the training set and evaluate on the noisy test set (`pixel_acc`); then `PCA(oracle_k)` + logistic (`pca_acc`). Also save `clean_acc` (logistic trained and evaluated on the clean data).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
sigma = 4
rng = np.random.default_rng(0)
X_tr_n = X_tr + rng.normal(0, sigma, X_tr.shape)
X_te_n = X_te + rng.normal(0, sigma, X_te.shape)
ks = [2, 5, 10, 15, 20, 25, 30, 40, 64]

## Your work

In [ ]:
mse_per_k = pd.Series(dtype=float)
noisy_mse = None
oracle_k = None

In [ ]:
mp_k = None
mp_mse = None
reduction = None

In [ ]:
pixel_acc = None
pca_acc = None
clean_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`p = PCA(k).fit(X_tr_n); R = p.inverse_transform(p.transform(X_te_n)); ((R - X_te) ** 2).mean()`. Criterion: `(PCA().fit(X_tr_n).explained_variance_ > threshold).sum()`. Classification: `make_pipeline(PCA(oracle_k), LogisticRegression(max_iter=3000)).fit(X_tr_n, y_tr).score(X_te_n, y_te)`.'):
    check(close(noisy_mse, ((X_te_n - X_te) ** 2).mean()) and close(noisy_mse, 16, rtol=0.1), "the noisy image has an error of about 16 (= sigma^2)", "np.mean((X_te_n - X_te) ** 2)")
    check(list(mse_per_k.index) == ks and mse_per_k[64] > 0.97 * noisy_mse, "with 64 components (no filter) the error stays that of the noise", "reconstruction with all the components = noisy image")
    check(oracle_k == mse_per_k.idxmin() and 10 <= oracle_k <= 20 and mse_per_k[oracle_k] < 0.5 * noisy_mse, "the best k is between 10 and 20 and halves the error (over 50%)", "idxmin of mse_per_k")
    check(20 <= mp_k <= 35 and mp_mse < 0.6 * noisy_mse and mp_mse > mse_per_k[oracle_k], "the noise-edge criterion gives k between 20 and 35: almost optimal, but more conservative than the oracle", "components with an eigenvalue above the threshold")
    check(close(reduction, 1 - mse_per_k[oracle_k] / noisy_mse) and reduction > 0.5, "the error is reduced by over 50%", "1 - oracle_mse / noisy_mse")
    check(pca_acc > pixel_acc + 0.02 and clean_acc > 0.95 and pixel_acc < 0.9, "on the noisy images the logistic on the components beats the one on the pixels by over 2 points (clean: above 0.95)", "compare pca_acc, pixel_acc, clean_acc")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_pca_to_remove_noise.ipynb`. Open it after you pass the check, or after you have really tried.